# Ranking LLMs on Structured Data Processing — 30 ML Algorithm Modules

This notebook ranks large language models on how accurately they **carry out the 30 core machine-learning
algorithms** by hand on structured data, from linear regression to genetic algorithms.

Every module turns rows of the AI4I 2020 predictive-maintenance dataset used elsewhere in this repo (machine
type, temperatures, rpm, torque, tool wear, failures) into a small, fully specified problem: "fit this line",
"run Lloyd's algorithm from these centroids", "apply one Q-learning update", "compute this attention layer".
The ground truth comes from a reference implementation (scikit-learn where it has the algorithm, NumPy
otherwise), so scoring is automatic. There are no hand labels and no LLM judge.

| Family | Modules |
|---|---|
| **Supervised** | 1 Linear Regression · 2 Logistic Regression · 3 Decision Tree · 4 Random Forest · 5 SVM · 6 k-NN · 7 Naive Bayes · 8 Gradient Boosting · 9 AdaBoost · 10 XGBoost |
| **Unsupervised** | 11 k-Means · 12 Hierarchical Clustering · 13 DBSCAN · 14 PCA · 15 t-SNE · 26 K-Means++ · 28 Isolation Forest |
| **Reinforcement learning** | 16 Q-Learning · 17 SARSA · 18 DQN · 19 Policy Gradient · 20 Actor-Critic · 29 Markov Decision Process |
| **Deep learning** | 21 ANN · 22 CNN · 23 RNN · 24 LSTM · 25 Transformer · 27 Autoencoder |
| **Evolutionary** | 30 Genetic Algorithm |

Notebook sections:

1. **Configuration**
2. **Data**: AI4I 2020 loader (local CSV → KaggleHub → seeded synthetic fallback)
3. **The 30 algorithm modules**: problem generator + reference solution for each
4. **Model registry & adapters**: Anthropic SDK plus any OpenAI-compatible endpoint, plus calibration baselines
5. **Execution engine**: parallel calls, response cache, latency / token / cost accounting
6. **Scoring**: final-JSON extraction and tolerance-aware comparison (partitions, sets, sign-invariant vectors)
7. **Ranking**: composite and family scores, bootstrap 95% CIs, rank probabilities, Bradley–Terry ratings, tiers
8. **Reporting**: leaderboard, per-module heatmap, cost-vs-quality chart, CSV / Markdown export

With no API keys set, only the calibration baselines run, which checks the pipeline end to end. Set keys
(Section 4) to rank real models. Responses are cached in `llm_bench_cache/`, so re-runs do not re-bill.

In [ ]:
# %pip install -q pandas numpy scipy scikit-learn matplotlib anthropic openai kagglehub

## 1 · Configuration

In [ ]:
import hashlib
import json
import math
import os
import random
import re
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Callable

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist, pdist
from sklearn.cluster import DBSCAN, AgglomerativeClustering, KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.naive_bayes import CategoricalNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier, export_text

SEED = 42
ITEMS_PER_MODULE = 5         # problems per algorithm module (30 modules → 150 requests per model)
MAX_WORKERS = 4              # concurrent requests per model
MAX_OUTPUT_TOKENS = 16000    # models may show their working before the final JSON
REQUEST_TIMEOUT_S = 600
BOOTSTRAP_SAMPLES = 1000
INCLUDE_BASELINES = True     # show calibration baselines in the leaderboard

# Relative weight of each module in the composite score (keyed by module number; default 1.0).
MODULE_WEIGHTS: dict[int, float] = {}

# Numeric answers count as correct within max(REL_TOL × |expected|, ABS_TOL).
REL_TOL, ABS_TOL = 0.02, 0.01

CACHE_DIR = Path("llm_bench_cache")
RESULTS_DIR = Path("llm_bench_results")
CACHE_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

## 2 · Data

Loads the AI4I 2020 predictive-maintenance dataset. Lookup order:

1. a local CSV (`AI4I_CSV_PATH` env var, or `ai4i2020.csv` / `predictive_maintenance.csv` next to the notebook);
2. KaggleHub (same dataset as the failure-prediction notebook in this repo);
3. a deterministic synthetic generator with the same schema and the published failure rules.

Columns are normalised to snake_case. `has_mode` marks rows that trigger at least one failure-mode rule. It is
used as the binary label in the classification modules because it is better balanced than `machine_failure`.

In [ ]:
CANONICAL_COLUMNS = {
    "udi": ["udi"],
    "product_id": ["product id", "product_id"],
    "type": ["type"],
    "air_temp_k": ["air temperature"],
    "process_temp_k": ["process temperature"],
    "rpm": ["rotational speed"],
    "torque_nm": ["torque"],
    "tool_wear_min": ["tool wear"],
    "machine_failure": ["machine failure", "target"],
    "twf": ["twf"], "hdf": ["hdf"], "pwf": ["pwf"], "osf": ["osf"], "rnf": ["rnf"],
}


def normalise_columns(df: pd.DataFrame) -> pd.DataFrame:
    rename = {}
    for col in df.columns:
        low = col.lower().strip()
        for canon, keys in CANONICAL_COLUMNS.items():
            if canon not in rename.values() and any(low == k or low.startswith(k) for k in keys):
                rename[col] = canon
                break
    df = df.rename(columns=rename)
    missing = {"product_id", "type", "air_temp_k", "process_temp_k", "rpm", "torque_nm", "tool_wear_min"} - set(df.columns)
    if missing:
        raise ValueError(f"Dataset is missing required columns: {sorted(missing)}")
    if "udi" not in df.columns:
        df.insert(0, "udi", range(1, len(df) + 1))
    for flag in ["machine_failure", "twf", "hdf", "pwf", "osf", "rnf"]:
        if flag not in df.columns:
            df[flag] = 0
    return df[list(CANONICAL_COLUMNS)]


# Failure-mode rules (AI4I 2020 documentation; TWF made deterministic so labels are well defined).
OSF_LIMIT = {"L": 11_000, "M": 12_000, "H": 13_000}


def failure_modes(row) -> list[str]:
    modes = []
    if row["tool_wear_min"] >= 200:
        modes.append("TWF")
    if (row["process_temp_k"] - row["air_temp_k"]) < 8.6 and row["rpm"] < 1380:
        modes.append("HDF")
    power_w = row["torque_nm"] * row["rpm"] * 2 * math.pi / 60
    if power_w < 3500 or power_w > 9000:
        modes.append("PWF")
    if row["tool_wear_min"] * row["torque_nm"] > OSF_LIMIT[row["type"]]:
        modes.append("OSF")
    return modes


def synthetic_ai4i(n: int = 10_000, seed: int = SEED) -> pd.DataFrame:
    """AI4I-2020-shaped data generated from the dataset's published distributions and failure rules."""
    rng = np.random.default_rng(seed)
    types = rng.choice(["L", "M", "H"], size=n, p=[0.6, 0.3, 0.1])
    air = np.round(rng.normal(300.0, 2.0, n), 1)
    process = np.round(air + 10.0 + rng.normal(0.0, 1.0, n), 1)
    rpm = np.clip(np.round(rng.normal(1540, 180, n)), 1168, 2886).astype(int)
    torque = np.round(np.clip(rng.normal(40.0, 10.0, n), 3.8, 76.6), 1)
    wear = rng.integers(0, 254, n)
    df = pd.DataFrame({
        "udi": np.arange(1, n + 1),
        "product_id": [f"{t}{rng.integers(10000, 99999)}" for t in types],
        "type": types, "air_temp_k": air, "process_temp_k": process,
        "rpm": rpm, "torque_nm": torque, "tool_wear_min": wear,
    })
    modes = df.apply(failure_modes, axis=1)
    for m in ["TWF", "HDF", "PWF", "OSF"]:
        df[m.lower()] = modes.apply(lambda s, m=m: int(m in s))
    df["rnf"] = (rng.random(n) < 0.001).astype(int)
    df["machine_failure"] = df[["twf", "hdf", "pwf", "osf", "rnf"]].max(axis=1)
    return df[list(CANONICAL_COLUMNS)]


def load_ai4i() -> tuple[pd.DataFrame, str]:
    candidates = [os.environ.get("AI4I_CSV_PATH"), "ai4i2020.csv", "predictive_maintenance.csv"]
    for path in filter(None, candidates):
        if Path(path).is_file():
            return normalise_columns(pd.read_csv(path)), f"local CSV ({path})"
    try:
        import kagglehub
        folder = Path(kagglehub.dataset_download("abdelazizsami/predictive-maintenance-dataset"))
        csv = next(folder.glob("*.csv"))
        return normalise_columns(pd.read_csv(csv)), f"KaggleHub ({csv.name})"
    except Exception as exc:  # no network, no kagglehub, or no credentials
        print(f"KaggleHub unavailable ({type(exc).__name__}); using synthetic AI4I-shaped data.")
        return synthetic_ai4i(), "synthetic (seeded)"


df, DATA_SOURCE = load_ai4i()
df["type"] = df["type"].astype(str).str.strip().str.upper()
df["has_mode"] = df.apply(lambda r: int(bool(failure_modes(r))), axis=1)

NUMERIC = ["air_temp_k", "process_temp_k", "rpm", "torque_nm", "tool_wear_min"]
MU, SD = df[NUMERIC].mean(), df[NUMERIC].std()

print(f"Data source: {DATA_SOURCE} — {len(df):,} rows, {df.has_mode.mean():.1%} trigger a failure-mode rule")
df.head()

## 3 · The 30 algorithm modules

Each module is a generator that draws a small problem from the data (seeded, so every model sees identical
items) and returns:

* `body`: the problem statement given to the model, with every convention spelled out (tie rules, `<=` vs
  `<`, smoothing, learning rates…) so there is exactly one correct answer;
* `answer`: the JSON shape the final answer must take;
* `expected`: the reference answer, computed with scikit-learn or NumPy;
* `meta`: scoring hints (`partition` keys compare clusterings label-invariantly, `set` keys ignore order,
  `sign_invariant` keys accept ±v).

Generators return `None` to reject a draw that would be ambiguous (ties, values on a decision boundary), and the
item is redrawn. Problems are sized to be solvable by careful hand computation.

In [ ]:
@dataclass
class Problem:
    body: str
    answer: str
    expected: dict
    meta: dict = field(default_factory=dict)


@dataclass
class Module:
    num: int
    name: str
    family: str
    generate: Callable[[np.random.Generator], Problem | None]

    @property
    def key(self) -> str:
        return f"{self.num:02d} {self.name}"


MODULES: dict[int, Module] = {}


def module(num: int, name: str, family: str):
    def register(fn):
        MODULES[num] = Module(num, name, family, fn)
        return fn
    return register


# ---------------------------------------------------------------- helpers
def to_py(o):
    """Convert numpy containers / scalars to plain JSON-serialisable Python."""
    if isinstance(o, dict):
        return {k: to_py(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [to_py(v) for v in o]
    if isinstance(o, np.ndarray):
        return to_py(o.tolist())
    if isinstance(o, np.bool_):
        return bool(o)
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, np.floating):
        return float(o)
    return o


def seed_of(rng: np.random.Generator) -> int:
    return int(rng.integers(2**31))


def sample(rng: np.random.Generator, n: int, pos_frac: float | None = None) -> pd.DataFrame:
    """Random rows; with pos_frac, a controlled share have has_mode = 1."""
    if pos_frac is None:
        return df.sample(n, random_state=seed_of(rng))
    k = int(round(n * pos_frac))
    pos = df[df.has_mode == 1].sample(k, random_state=seed_of(rng))
    neg = df[df.has_mode == 0].sample(n - k, random_state=seed_of(rng))
    return pd.concat([pos, neg]).sample(frac=1, random_state=seed_of(rng))


def zs(rows: pd.DataFrame, col: str, d: int = 1) -> np.ndarray:
    """Standardised column (z-score against the full dataset), rounded."""
    return np.round((rows[col].to_numpy() - MU[col]) / SD[col], d)


def js(x) -> str:
    return json.dumps(to_py(x))


def indexed(points) -> str:
    return "\n".join(f"  {i}: {js(p)}" for i, p in enumerate(to_py(points)))


def partition(labels) -> list[list[int]]:
    groups: dict[Any, list[int]] = {}
    for i, lab in enumerate(labels):
        groups.setdefault(lab, []).append(i)
    return sorted(sorted(g) for g in groups.values())


def softmax(x):
    e = np.exp(np.asarray(x, float) - np.max(x))
    return e / e.sum()


def sigmoid(x):
    return 1 / (1 + np.exp(-np.asarray(x, float)))


def midpoints(values) -> np.ndarray:
    v = np.unique(values)
    return (v[:-1] + v[1:]) / 2


def gini(y) -> float:
    if len(y) == 0:
        return 0.0
    p = np.bincount(y, minlength=2) / len(y)
    return 1 - float((p ** 2).sum())

### Supervised learning (modules 1–10)

In [ ]:
@module(1, "Linear Regression", "Supervised")
def m01(rng):
    rows = sample(rng, 6)
    x, y = np.round(rows.rpm.to_numpy() / 100, 1), rows.torque_nm.to_numpy()
    if np.ptp(x) < 1:
        return None
    lr = LinearRegression().fit(x[:, None], y)
    x0 = round(float(rng.uniform(x.min(), x.max())), 1)
    return Problem(
        f"Fit an ordinary least-squares line y = slope·x + intercept to these points (x = rpm/100, y = torque_nm):\n"
        f"{js(np.c_[x, y])}\nThen predict y at x = {x0}.",
        '{"slope": number, "intercept": number, "prediction": number}',
        {"slope": lr.coef_[0], "intercept": lr.intercept_, "prediction": lr.predict([[x0]])[0]})


@module(2, "Logistic Regression", "Supervised")
def m02(rng):
    train = sample(rng, 300, 0.5)
    clf = LogisticRegression().fit(np.c_[zs(train, "torque_nm", 2), zs(train, "tool_wear_min", 2)], train.has_mode)
    w, b = np.round(clf.coef_[0], 2), round(float(clf.intercept_[0]), 2)
    q = sample(rng, 4)
    xq = np.c_[zs(q, "torque_nm"), zs(q, "tool_wear_min")]
    logit = xq @ w + b
    if np.any(np.abs(logit) < 0.05):
        return None
    p = sigmoid(logit)
    return Problem(
        f"A logistic-regression failure model uses standardised torque x1 and standardised tool wear x2 with "
        f"w1 = {w[0]}, w2 = {w[1]}, bias b = {b}. p = 1 / (1 + e^-(b + w1·x1 + w2·x2)); predict class 1 when p >= 0.5.\n"
        f"Rows [x1, x2]:\n{indexed(xq)}",
        '{"probabilities": [number per row], "classes": [0 or 1 per row]}',
        {"probabilities": p, "classes": (p >= 0.5).astype(int)})


@module(3, "Decision Tree", "Supervised")
def m03(rng):
    rows = sample(rng, 10, 0.5)
    x, y = rows.tool_wear_min.to_numpy(), rows.has_mode.to_numpy()
    if len(np.unique(x)) < 10:
        return None
    cands = midpoints(x)
    scores = np.array([(np.sum(x <= t) * gini(y[x <= t]) + np.sum(x > t) * gini(y[x > t])) / len(y) for t in cands])
    best = int(scores.argmin())
    if np.sum(np.isclose(scores, scores[best])) > 1:
        return None
    stump = DecisionTreeClassifier(max_depth=1).fit(x[:, None], y)   # cross-check against scikit-learn
    if not math.isclose(stump.tree_.threshold[0], cands[best]):
        return None
    return Problem(
        "Find the best single split for a depth-1 decision tree (Gini impurity) on feature tool_wear_min predicting "
        "label (1 = failure mode). Candidate thresholds are midpoints between consecutive distinct sorted values; rows "
        "with value <= threshold go left. Weighted Gini = (n_left·Gini_left + n_right·Gini_right) / n, with "
        f"Gini = 1 − Σ p_k².\nRows [tool_wear_min, label]:\n{js(np.c_[x, y])}",
        '{"threshold": number, "weighted_gini": number}',
        {"threshold": cands[best], "weighted_gini": scores[best]})


@module(4, "Random Forest", "Supervised")
def m04(rng):
    feats = ["torque_nm", "tool_wear_min"]
    train = sample(rng, 200, 0.5)
    rf = RandomForestClassifier(n_estimators=5, max_depth=2, random_state=seed_of(rng)).fit(train[feats].to_numpy(), train.has_mode)
    q = sample(rng, 4)[feats].to_numpy()
    votes = np.array([est.predict(q) for est in rf.estimators_]).T.astype(int)   # rows × trees
    trees = "\n\n".join(f"Tree {i + 1}:\n" + export_text(est, feature_names=feats, decimals=2)
                        for i, est in enumerate(rf.estimators_))
    return Problem(
        "A random forest has 5 trees (class 1 = failure mode). At each node follow the first branch when the condition "
        "holds. The forest prediction is the majority vote of the 5 trees.\n\n"
        f"{trees}\nRows [torque_nm, tool_wear_min]:\n{indexed(q)}",
        '{"tree_votes": [[5 votes (0/1)] per row], "predictions": [0 or 1 per row]}',
        {"tree_votes": votes, "predictions": (votes.sum(axis=1) >= 3).astype(int)})


@module(5, "Support Vector Machine", "Supervised")
def m05(rng):
    train = sample(rng, 300, 0.5)
    svm = LinearSVC(dual=False).fit(np.c_[zs(train, "torque_nm", 2), zs(train, "rpm", 2)], train.has_mode)
    w, b = np.round(svm.coef_[0], 2), round(float(svm.intercept_[0]), 2)
    q = sample(rng, 5)
    xq = np.c_[zs(q, "torque_nm"), zs(q, "rpm")]
    f = xq @ w + b
    if np.any(np.abs(f) < 0.03) or np.any(np.abs(np.abs(f) - 1) < 0.03):
        return None
    return Problem(
        f"A linear SVM over standardised torque x1 and standardised rpm x2 has w = {js(w)} and b = {b}. "
        "Decision value f(x) = w·x + b; class 1 if f >= 0 else 0; a point is inside the margin if |f| < 1.\n"
        f"Rows [x1, x2]:\n{indexed(xq)}",
        '{"decision_values": [number per row], "classes": [0/1 per row], "inside_margin": [true/false per row]}',
        {"decision_values": f, "classes": (f >= 0).astype(int), "inside_margin": np.abs(f) < 1})


@module(6, "k-Nearest Neighbors", "Supervised")
def m06(rng):
    train, q = sample(rng, 10, 0.5), sample(rng, 3)
    x = np.c_[zs(train, "torque_nm"), zs(train, "tool_wear_min")]
    y = train.has_mode.to_numpy()
    xq = np.c_[zs(q, "torque_nm"), zs(q, "tool_wear_min")]
    d = cdist(xq, x)
    srt = np.sort(d, axis=1)
    if np.any(srt[:, 3] - srt[:, 2] < 1e-6):
        return None
    labels = KNeighborsClassifier(3).fit(x, y).predict(xq)
    return Problem(
        "Classify each query with k-NN (k = 3, Euclidean distance, majority vote) over standardised [torque, tool_wear]."
        f"\nTraining points [x1, x2] with labels:\n" + "\n".join(f"  {i}: {js(p)} label {l}" for i, (p, l) in enumerate(zip(x, y)))
        + f"\nQueries:\n{indexed(xq)}",
        '{"neighbors": [[3 training indices, ascending] per query], "labels": [0/1 per query]}',
        {"neighbors": [sorted(np.argsort(r)[:3]) for r in d], "labels": labels})


NB_CATS = {"type": ["L", "M", "H"], "wear_band": ["low", "mid", "high"], "temp_gap": ["small", "normal"]}


def nb_features(rows):
    wear = np.where(rows.tool_wear_min < 100, "low", np.where(rows.tool_wear_min < 200, "mid", "high"))
    gap = np.where(rows.process_temp_k - rows.air_temp_k < 8.6, "small", "normal")
    return np.c_[rows.type.to_numpy(), wear, gap]


@module(7, "Naive Bayes", "Supervised")
def m07(rng):
    train = sample(rng, 12, 0.5)
    xs, y = nb_features(train), train.has_mode.to_numpy()
    enc = lambda X: np.array([[NB_CATS[c].index(v) for c, v in zip(NB_CATS, row)] for row in X])
    nb = CategoricalNB(alpha=1.0, min_categories=[len(v) for v in NB_CATS.values()]).fit(enc(xs), y)
    q = np.array([[rng.choice(v) for v in NB_CATS.values()] for _ in range(2)])
    p = nb.predict_proba(enc(q))[:, 1]
    rows = "\n".join(f"  {js(list(r))} → {l}" for r, l in zip(xs, y))
    return Problem(
        "Train a categorical Naive Bayes classifier on [type, wear_band, temp_gap] → failure (1/0). Priors are class "
        "frequencies (no smoothing). Likelihoods use Laplace smoothing: P(v | c) = (count(v, c) + 1) / (count(c) + K), "
        f"where K is the number of possible values of that feature: {js(NB_CATS)}.\nTraining rows:\n{rows}\n"
        f"Queries:\n{indexed(q)}\nReturn the posterior P(failure = 1 | query) for each query.",
        '{"p_failure": [number per query]}',
        {"p_failure": p})


@module(8, "Gradient Boosting", "Supervised")
def m08(rng):
    rows = sample(rng, 6)
    x, y = rows.tool_wear_min.to_numpy(), rows.torque_nm.to_numpy()
    if len(np.unique(x)) < 6:
        return None
    t = float(rng.choice(midpoints(x)))
    f0 = y.mean()
    r = y - f0
    left = x <= t
    lv, rv = r[left].mean(), r[~left].mean()
    f1 = f0 + 0.5 * np.where(left, lv, rv)
    return Problem(
        "Run one round of gradient boosting for regression with squared loss. F0 = mean(y). Fit residuals r = y − F0 "
        f"with a stump that splits tool_wear_min at {t} (<= goes left); each leaf value is the mean residual in it. "
        f"Learning rate 0.5: F1 = F0 + 0.5·leaf_value.\nRows [tool_wear_min, y = torque_nm]:\n{js(np.c_[x, y])}",
        '{"F0": number, "leaf_left": number, "leaf_right": number, "F1": [number per row, input order]}',
        {"F0": f0, "leaf_left": lv, "leaf_right": rv, "F1": f1})


@module(9, "AdaBoost", "Supervised")
def m09(rng):
    rows = sample(rng, 8, 0.5)
    x, y = rows.tool_wear_min.to_numpy(), np.where(rows.has_mode == 1, 1, -1)
    if len(np.unique(x)) < 2:
        return None
    t = float(rng.choice(midpoints(x)))
    h = np.where(x > t, 1, -1)
    w = np.full(len(x), 1 / len(x))
    err = w[h != y].sum()
    if not 0 < err < 0.5:
        return None
    alpha = 0.5 * math.log((1 - err) / err)
    w_new = w * np.exp(-alpha * y * h)
    return Problem(
        "Run one AdaBoost round. Start with equal sample weights. The weak learner predicts +1 if tool_wear_min > "
        f"{t}, else −1. Weighted error ε = Σ w_i over misclassified rows; α = ½·ln((1 − ε)/ε); "
        "w_i ← w_i·exp(−α·y_i·h_i), then normalise to sum to 1.\n"
        f"Rows [tool_wear_min, y]:\n{js(np.c_[x, y])}",
        '{"error": number, "alpha": number, "weights": [number per row]}',
        {"error": err, "alpha": alpha, "weights": w_new / w_new.sum()})


@module(10, "XGBoost", "Supervised")
def m10(rng):
    rows = sample(rng, 6)
    x, y = rows.tool_wear_min.to_numpy(), rows.torque_nm.to_numpy()
    if len(np.unique(x)) < 6:
        return None
    pred = np.round(y.mean() + rng.normal(0, 3, 6), 1)
    t = float(rng.choice(midpoints(x)))
    g, h, lam, gamma = pred - y, np.ones(6), 1.0, 0.5
    left = x <= t
    GL, GR, HL, HR = g[left].sum(), g[~left].sum(), h[left].sum(), h[~left].sum()
    gain = 0.5 * (GL**2 / (HL + lam) + GR**2 / (HR + lam) - (GL + GR) ** 2 / (HL + HR + lam)) - gamma
    return Problem(
        "XGBoost split evaluation with loss ½(y − ŷ)², so gradient g = ŷ − y and hessian h = 1. λ = 1, γ = 0.5. "
        f"Split tool_wear_min at {t} (<= goes left). Leaf weight w = −G/(H + λ); "
        "gain = ½[G_L²/(H_L+λ) + G_R²/(H_R+λ) − (G_L+G_R)²/(H_L+H_R+λ)] − γ.\n"
        f"Rows [tool_wear_min, y, current prediction ŷ]:\n{js(np.c_[x, y, pred])}",
        '{"G_left": number, "G_right": number, "w_left": number, "w_right": number, "gain": number}',
        {"G_left": GL, "G_right": GR, "w_left": -GL / (HL + lam), "w_right": -GR / (HR + lam), "gain": gain})

### Unsupervised learning (modules 11–15, 26, 28)

In [ ]:
def int_points(rng, n):
    """Integer 2-D sensor points [round(torque_nm), tool_wear_min // 10]."""
    rows = sample(rng, n)
    return np.c_[np.round(rows.torque_nm.to_numpy()), rows.tool_wear_min.to_numpy() // 10].astype(int)


def lloyd(x, centres, iters=100):
    c = centres.astype(float)
    for _ in range(iters):
        lab = cdist(x, c).argmin(axis=1)
        if len(np.unique(lab)) < len(c):
            return None, None
        new = np.array([x[lab == k].mean(axis=0) for k in range(len(c))])
        if np.allclose(new, c):
            break
        c = new
    return lab, c


@module(11, "k-Means Clustering", "Unsupervised")
def m11(rng):
    x = int_points(rng, 8)
    init_idx = rng.choice(8, 2, replace=False)
    lab, cent = lloyd(x, x[init_idx])
    if lab is None:
        return None
    d = cdist(x, cent)
    if np.any(np.abs(d[:, 0] - d[:, 1]) < 1e-6):
        return None
    km = KMeans(2, init=x[init_idx].astype(float), n_init=1, algorithm="lloyd").fit(x)   # cross-check
    if not np.array_equal(km.labels_, lab):
        return None
    return Problem(
        "Run k-means (Lloyd's algorithm, Euclidean distance) to convergence with k = 2, starting from centroid 0 = "
        f"point {init_idx[0]} and centroid 1 = point {init_idx[1]}.\nPoints [torque, wear/10]:\n{indexed(x)}",
        '{"labels": [0 or 1 per point], "centroids": [[x, y] for centroid 0, [x, y] for centroid 1]}',
        {"labels": lab, "centroids": cent})


@module(12, "Hierarchical Clustering", "Unsupervised")
def m12(rng):
    rows = sample(rng, 7)
    x = np.c_[rows.torque_nm.to_numpy(), rows.tool_wear_min.to_numpy() / 10]
    d = np.round(pdist(x), 6)
    if len(np.unique(d)) < len(d):
        return None
    linkage = str(rng.choice(["single", "complete"]))
    parts = {k: partition(AgglomerativeClustering(n_clusters=k, linkage=linkage).fit_predict(x)) for k in (2, 3)}
    return Problem(
        f"Run agglomerative hierarchical clustering with {linkage} linkage and Euclidean distance. Report the clusters "
        f"when the dendrogram is cut at 3 clusters and at 2 clusters.\nPoints [torque, wear/10]:\n{indexed(x)}",
        '{"clusters_k3": [[point indices], ...], "clusters_k2": [[point indices], ...]}',
        {"clusters_k3": parts[3], "clusters_k2": parts[2]},
        {"partition": ["clusters_k3", "clusters_k2"]})


@module(13, "DBSCAN", "Unsupervised")
def m13(rng):
    centres = rng.integers(0, 25, (2, 2))
    if np.linalg.norm(centres[0] - centres[1]) < 8:
        return None
    blobs = [c + rng.integers(-2, 3, (4, 2)) for c in centres]
    x = np.vstack(blobs + [rng.integers(0, 28, (3, 2))])
    x = x[rng.permutation(len(x))]
    eps, min_samples = 2.5, 3
    db = DBSCAN(eps=eps, min_samples=min_samples).fit(x)
    core = np.zeros(len(x), bool)
    core[db.core_sample_indices_] = True
    d = cdist(x, x)
    for i in np.where(~core & (db.labels_ >= 0))[0]:      # border point reachable from two clusters is ambiguous
        if len({db.labels_[j] for j in np.where(core & (d[i] <= eps))[0]}) > 1:
            return None
    if db.labels_.max() < 0:
        return None
    clustered = np.where(db.labels_ >= 0)[0]
    return Problem(
        f"Run DBSCAN with eps = {eps} and min_samples = {min_samples} (Euclidean distance; a point counts itself as a "
        "neighbour, and points at distance exactly eps are neighbours).\n"
        f"Points [torque bucket, wear bucket]:\n{indexed(x)}",
        '{"clusters": [[point indices], ...], "noise": [point indices], "core_points": [point indices]}',
        {"clusters": [sorted(g) for g in partition(db.labels_) if db.labels_[g[0]] >= 0] if len(clustered) else [],
         "noise": sorted(np.where(db.labels_ < 0)[0]), "core_points": sorted(np.where(core)[0])},
        {"partition": ["clusters"], "set": ["noise", "core_points"]})


@module(14, "Principal Component Analysis", "Unsupervised")
def m14(rng):
    rows = sample(rng, 6)
    x = np.c_[zs(rows, "torque_nm"), zs(rows, "rpm")]
    pca = PCA(2).fit(x)
    ratio = pca.explained_variance_ratio_[0]
    if ratio < 0.6:
        return None
    return Problem(
        "Run PCA on these 2-D points (centre the data, do not scale). Give the first principal component as a unit "
        "vector and the fraction of total variance it explains.\n"
        f"Points [standardised torque, standardised rpm]:\n{indexed(x)}",
        '{"pc1": [number, number], "explained_variance_ratio": number}',
        {"pc1": pca.components_[0], "explained_variance_ratio": ratio},
        {"sign_invariant": ["pc1"]})


@module(15, "t-SNE", "Unsupervised")
def m15(rng):
    rows = sample(rng, 4)
    x = np.c_[zs(rows, "torque_nm"), zs(rows, "tool_wear_min")]
    y = np.round(rng.uniform(-2, 2, (4, 2)), 1)
    a = np.exp(-((x[1:] - x[0]) ** 2).sum(axis=1) / 2)          # sigma_0 = 1
    num = 1 / (1 + cdist(y, y, "sqeuclidean"))
    np.fill_diagonal(num, 0)
    return Problem(
        "Compute two t-SNE quantities.\n"
        "(a) High-dimensional affinities for point 0 with σ₀ = 1: p_{j|0} = exp(−‖x_0 − x_j‖² / 2σ₀²) / Σ_{k≠0} exp(−‖x_0 − x_k‖² / 2σ₀²), for j = 1, 2, 3.\n"
        "(b) The low-dimensional affinity q_01 = (1 + ‖y_0 − y_1‖²)⁻¹ / Σ_{k≠l} (1 + ‖y_k − y_l‖²)⁻¹ (sum over all ordered pairs).\n"
        f"High-dimensional points x:\n{indexed(x)}\nLow-dimensional embedding y:\n{indexed(y)}",
        '{"p_given_0": [p_1|0, p_2|0, p_3|0], "q_01": number}',
        {"p_given_0": a / a.sum(), "q_01": num[0, 1] / num.sum()})


@module(26, "K-Means++", "Unsupervised")
def m26(rng):
    x = int_points(rng, 6)
    first = int(rng.integers(6))
    centres, d2_first, picks, us = [first], None, [], []
    for _ in range(2):
        d2 = cdist(x, x[centres], "sqeuclidean").min(axis=1)
        if d2_first is None:
            d2_first = d2
        if d2.sum() == 0:
            return None
        cum = np.cumsum(d2 / d2.sum())
        u = round(float(rng.uniform(0.05, 0.95)), 2)
        if np.any(np.abs(cum - u) < 0.02):
            return None
        pick = int(np.argmax(cum >= u))
        centres.append(pick); picks.append(pick); us.append(u)
    return Problem(
        f"Run K-Means++ seeding for k = 3. The first centre is point {first}. For each next centre compute D(x)² = "
        "squared Euclidean distance from each point to its nearest chosen centre, form probabilities D(x)² / ΣD², and "
        "pick the first point (in index order) whose cumulative probability is >= u. "
        f"Use u = {us[0]} for the second centre and u = {us[1]} for the third.\nPoints:\n{indexed(x)}",
        '{"d2_from_first_centre": [number per point], "second_centre": index, "third_centre": index}',
        {"d2_from_first_centre": d2_first, "second_centre": picks[0], "third_centre": picks[1]})


EULER_GAMMA = 0.5772156649


def c_factor(n: int) -> float:
    if n <= 1:
        return 0.0
    if n == 2:
        return 1.0
    return 2 * (math.log(n - 1) + EULER_GAMMA) - 2 * (n - 1) / n


def build_itree(rng, x, depth, max_depth, names):
    if depth == max_depth or len(x) <= 1:
        return {"size": len(x)}
    for _ in range(10):
        f = int(rng.integers(x.shape[1]))
        lo, hi = x[:, f].min(), x[:, f].max()
        if lo == hi:
            continue
        t = round(float(rng.uniform(lo, hi)), 1)
        left = x[:, f] <= t
        if 0 < left.sum() < len(x):
            return {"feature": f, "threshold": t,
                    "left": build_itree(rng, x[left], depth + 1, max_depth, names),
                    "right": build_itree(rng, x[~left], depth + 1, max_depth, names)}
    return {"size": len(x)}


def render_itree(node, names, indent="  "):
    if "size" in node:
        return f"{indent}leaf (size={node['size']})"
    return (f"{indent}if {names[node['feature']]} <= {node['threshold']}:\n{render_itree(node['left'], names, indent + '    ')}\n"
            f"{indent}else:\n{render_itree(node['right'], names, indent + '    ')}")


def path_length(node, p, depth=0):
    if "size" in node:
        return depth + c_factor(node["size"])
    return path_length(node["left"] if p[node["feature"]] <= node["threshold"] else node["right"], p, depth + 1)


@module(28, "Isolation Forest", "Unsupervised")
def m28(rng):
    names = ["torque_nm", "tool_wear_min"]
    train = sample(rng, 8)[names].to_numpy()
    trees = [build_itree(rng, train, 0, 3, names) for _ in range(3)]
    q = np.array([sample(rng, 1)[names].to_numpy()[0], [round(float(rng.uniform(60, 75)), 1), int(rng.integers(220, 254))]])
    if np.any([np.isclose(p[t["feature"]], t["threshold"]) for t in trees if "feature" in t for p in q]):
        return None
    h = np.array([[path_length(t, p) for t in trees] for p in q])
    scores = 2 ** (-h.mean(axis=1) / c_factor(len(train)))
    text = "\n\n".join(f"Tree {i + 1}:\n{render_itree(t, names)}" for i, t in enumerate(trees))
    return Problem(
        "Score points with this isolation forest (3 trees, each built on a sample of ψ = 8 points). Path length "
        "h(x) = number of edges from the root to the leaf + c(size), where c(n) = 2·(ln(n − 1) + 0.5772156649) − 2(n − 1)/n "
        "for n > 2, c(2) = 1, c(1) = 0. Anomaly score s = 2^(−mean_h / c(8)).\n\n"
        f"{text}\n\nPoints [torque_nm, tool_wear_min]:\n{indexed(q)}",
        '{"path_lengths": [[h per tree] per point], "anomaly_scores": [number per point]}',
        {"path_lengths": h, "anomaly_scores": scores})

### Reinforcement learning (modules 16–20, 29)

All RL modules use the same maintenance environment: machine health states `good`, `worn`, `critical` and
actions `run`, `maintain`.

In [ ]:
STATES, ACTIONS = ["good", "worn", "critical"], ["run", "maintain"]


def q_table_text(q):
    return "\n".join(f"  {s}: run = {q[i, 0]}, maintain = {q[i, 1]}" for i, s in enumerate(STATES))


def random_transitions(rng, n, with_next_action=False):
    out = []
    for _ in range(n):
        t = {"s": int(rng.integers(3)), "a": int(rng.integers(2)), "r": int(rng.integers(-10, 11)), "s2": int(rng.integers(3))}
        if with_next_action:
            t["a2"] = int(rng.integers(2))
        out.append(t)
    return out


def describe(t):
    s = f"(s = {STATES[t['s']]}, a = {ACTIONS[t['a']]}, r = {t['r']}, s' = {STATES[t['s2']]}"
    return s + (f", a' = {ACTIONS[t['a2']]})" if "a2" in t else ")")


@module(16, "Q-Learning", "Reinforcement")
def m16(rng):
    q = rng.integers(0, 11, (3, 2)).astype(float)
    ts = random_transitions(rng, 3)
    out = q.copy()
    for t in ts:
        out[t["s"], t["a"]] += 0.5 * (t["r"] + 0.9 * out[t["s2"]].max() - out[t["s"], t["a"]])
    return Problem(
        "Apply these Q-learning updates in order with α = 0.5 and γ = 0.9: "
        "Q(s,a) ← Q(s,a) + α[r + γ·max_a' Q(s',a') − Q(s,a)].\n"
        f"Initial Q-table:\n{q_table_text(q)}\nTransitions:\n" + "\n".join(f"  {i + 1}. {describe(t)}" for i, t in enumerate(ts)),
        '{"Q": [[run, maintain] for good, [..] for worn, [..] for critical]}',
        {"Q": out})


@module(17, "SARSA", "Reinforcement")
def m17(rng):
    q = rng.integers(0, 11, (3, 2)).astype(float)
    ts = random_transitions(rng, 3, with_next_action=True)
    out = q.copy()
    for t in ts:
        out[t["s"], t["a"]] += 0.5 * (t["r"] + 0.9 * out[t["s2"], t["a2"]] - out[t["s"], t["a"]])
    return Problem(
        "Apply these SARSA updates in order with α = 0.5 and γ = 0.9: "
        "Q(s,a) ← Q(s,a) + α[r + γ·Q(s',a') − Q(s,a)].\n"
        f"Initial Q-table:\n{q_table_text(q)}\nTransitions:\n" + "\n".join(f"  {i + 1}. {describe(t)}" for i, t in enumerate(ts)),
        '{"Q": [[run, maintain] for good, [..] for worn, [..] for critical]}',
        {"Q": out})


@module(18, "Deep Q-Network", "Reinforcement")
def m18(rng):
    r = rng.integers(-10, 11, 4)
    done = rng.random(4) < 0.25
    q_next = np.round(rng.uniform(-5, 15, (4, 2)), 1)
    q_sa = np.round(rng.uniform(-5, 15, 4), 1)
    y = r + 0.99 * (~done) * q_next.max(axis=1)
    rows = "\n".join(f"  {i}: r = {r[i]}, done = {str(bool(done[i])).lower()}, target-net Q(s',·) = {js(q_next[i])}, "
                     f"online Q(s,a) = {q_sa[i]}" for i in range(4))
    return Problem(
        "Compute DQN training targets and loss for this minibatch with γ = 0.99. Target y = r if done, else "
        "r + γ·max_a' Q_target(s',a'). Loss = mean of (y − Q_online(s,a))².\n" + rows,
        '{"targets": [number per transition], "loss": number}',
        {"targets": y, "loss": float(np.mean((y - q_sa) ** 2))})


@module(19, "Policy Gradient", "Reinforcement")
def m19(rng):
    rewards = rng.integers(-5, 11, 4)
    returns = np.array([sum(0.9 ** k * rewards[t + k] for k in range(4 - t)) for t in range(4)])
    theta = np.round(rng.uniform(-1, 1, 2), 1)
    a0 = int(rng.integers(2))
    grad = np.eye(2)[a0] - softmax(theta)
    return Problem(
        f"REINFORCE with γ = 0.9. An episode gives rewards r_0..r_3 = {js(rewards)}. Compute the returns "
        "G_t = Σ_k γ^k·r_{t+k}. At the first step the policy is softmax over logits θ = "
        f"{js(theta)} for actions [run, maintain], and action {ACTIONS[a0]} was taken. Update θ ← θ + α·G_0·∇_θ log π(a_0) "
        "with α = 0.1, where ∇_θ log π(a) = onehot(a) − softmax(θ).",
        '{"returns": [G_0, G_1, G_2, G_3], "theta_new": [number, number]}',
        {"returns": returns, "theta_new": theta + 0.1 * returns[0] * grad})


@module(20, "Actor-Critic", "Reinforcement")
def m20(rng):
    v, v2 = np.round(rng.uniform(0, 10, 2), 1)
    r = int(rng.integers(-5, 11))
    theta = np.round(rng.uniform(-1, 1, 2), 1)
    a = int(rng.integers(2))
    delta = r + 0.9 * v2 - v
    return Problem(
        f"One-step actor-critic update with γ = 0.9. Critic: V(s) = {v}, V(s') = {v2}; reward r = {r}. "
        f"TD error δ = r + γV(s') − V(s); critic update V(s) ← V(s) + 0.1·δ. Actor: softmax policy with logits θ = {js(theta)} "
        f"for [run, maintain]; action {ACTIONS[a]} was taken; update θ ← θ + 0.1·δ·(onehot(a) − softmax(θ)).",
        '{"td_error": number, "v_new": number, "theta_new": [number, number]}',
        {"td_error": delta, "v_new": v + 0.1 * delta, "theta_new": theta + 0.1 * delta * (np.eye(2)[a] - softmax(theta))})


@module(29, "Markov Decision Process", "Reinforcement")
def m29(rng):
    P = np.array([[rng.multinomial(10, rng.dirichlet(np.ones(3))) / 10 for _ in range(2)] for _ in range(3)])  # [s][a][s']
    R = rng.integers(-10, 11, (3, 2)).astype(float)
    gamma = 0.9
    v1 = R.max(axis=1)
    q2 = R + gamma * P @ v1
    v2 = q2.max(axis=1)
    q3 = R + gamma * P @ v2
    for q in (q2, q3):
        srt = np.sort(q, axis=1)
        if np.any(srt[:, 1] - srt[:, 0] < 0.05):
            return None
    lines = []
    for s in range(3):
        for a in range(2):
            lines.append(f"  {STATES[s]}, {ACTIONS[a]}: reward {R[s, a]:g}, P(s') over [good, worn, critical] = {js(P[s, a])}")
    return Problem(
        "Run synchronous value iteration on this MDP with γ = 0.9 starting from V0 = 0 for all states: "
        "V_{k+1}(s) = max_a [R(s,a) + γ·Σ_s' P(s'|s,a)·V_k(s')]. Report V1 and V2, and the greedy policy with respect "
        "to V2 (argmax_a of R(s,a) + γ·Σ P·V2).\n" + "\n".join(lines),
        '{"V1": [good, worn, critical], "V2": [good, worn, critical], "policy": ["run" or "maintain" per state]}',
        {"V1": v1, "V2": v2, "policy": [ACTIONS[i] for i in q3.argmax(axis=1)]})

### Deep learning (modules 21–25, 27)

In [ ]:
def w1(rng, shape, lo=-1.0, hi=1.0):
    return np.round(rng.uniform(lo, hi, shape), 1) + 0.0   # + 0.0 turns -0.0 into 0.0


@module(21, "Artificial Neural Network", "Deep learning")
def m21(rng):
    rows = sample(rng, 3)
    x = np.c_[zs(rows, "air_temp_k"), zs(rows, "torque_nm"), zs(rows, "tool_wear_min")]
    W, b, v, c = w1(rng, (2, 3)), w1(rng, 2), w1(rng, 2, -2, 2), w1(rng, None)
    hidden = np.maximum(0, x @ W.T + b)
    return Problem(
        "Forward pass of a 3-2-1 neural network: hidden = ReLU(W·x + b), output = sigmoid(v·hidden + c).\n"
        f"W = {js(W)} (rows = hidden units), b = {js(b)}, v = {js(v)}, c = {c}.\n"
        f"Inputs [standardised air temp, torque, tool wear]:\n{indexed(x)}",
        '{"hidden": [[h1, h2] per input], "outputs": [number per input]}',
        {"hidden": hidden, "outputs": sigmoid(hidden @ v + c)})


@module(22, "Convolutional Neural Network", "Deep learning")
def m22(rng):
    img = rng.integers(0, 10, (5, 5))
    k = rng.integers(-1, 2, (3, 3))
    conv = np.array([[(img[i:i + 3, j:j + 3] * k).sum() for j in range(3)] for i in range(3)])
    act = np.maximum(conv, 0)
    pooled = np.array([[act[i:i + 2, j:j + 2].max() for j in range(2)] for i in range(2)])
    return Problem(
        "A 5×5 sensor heat-map (rows = sensors, columns = time steps, quantised 0–9) passes through a conv layer: "
        "3×3 kernel, stride 1, no padding, no bias, cross-correlation (no kernel flip) → 3×3 map; then ReLU; then 2×2 "
        f"max-pooling with stride 1 → 2×2.\nInput = {js(img)}\nKernel = {js(k)}",
        '{"conv_output": 3×3 list of rows (before ReLU), "pooled": 2×2 list of rows}',
        {"conv_output": conv, "pooled": pooled})


@module(23, "Recurrent Neural Network", "Deep learning")
def m23(rng):
    x = zs(sample(rng, 4).sort_values("tool_wear_min"), "tool_wear_min")
    wx, wh, b = w1(rng, 3)
    h, hs = 0.0, []
    for xt in x:
        h = math.tanh(wx * xt + wh * h + b)
        hs.append(h)
    return Problem(
        f"A scalar Elman RNN reads a sequence of standardised tool-wear readings x = {js(x)}. "
        f"h_t = tanh(w_x·x_t + w_h·h_(t−1) + b) with h_0 = 0, w_x = {wx}, w_h = {wh}, b = {b}.",
        '{"hidden_states": [h_1, h_2, h_3, h_4]}',
        {"hidden_states": hs})


@module(24, "Long Short-Term Memory", "Deep learning")
def m24(rng):
    x = zs(sample(rng, 2), "torque_nm")
    p = {g: w1(rng, 3) for g in ["f", "i", "o", "g"]}   # [w_x, w_h, b]
    h, c, hs, cs = 0.0, 0.0, [], []
    for xt in x:
        pre = {g: p[g][0] * xt + p[g][1] * h + p[g][2] for g in p}
        f, i, o = (float(sigmoid(pre[g])) for g in "fio")
        c = f * c + i * math.tanh(pre["g"])
        h = o * math.tanh(c)
        hs.append(h); cs.append(c)
    return Problem(
        f"Run a scalar LSTM over x = {js(x)} (standardised torque) with h_0 = c_0 = 0. For each gate "
        "z = w_x·x_t + w_h·h_(t−1) + b; f, i, o = sigmoid(z); g = tanh(z); c_t = f·c_(t−1) + i·g; h_t = o·tanh(c_t).\n"
        "Parameters [w_x, w_h, b]: " + ", ".join(f"{g} = {js(v)}" for g, v in p.items()),
        '{"h": [h_1, h_2], "c": [c_1, c_2]}',
        {"h": hs, "c": cs})


@module(25, "Transformer", "Deep learning")
def m25(rng):
    Q, K, V = w1(rng, (3, 2)), w1(rng, (3, 2)), w1(rng, (3, 2))
    causal = bool(rng.random() < 0.5)
    scores = Q @ K.T / math.sqrt(2)
    if causal:
        scores = np.where(np.tril(np.ones((3, 3))) == 1, scores, -np.inf)
    w = np.array([softmax(r) for r in scores])
    return Problem(
        "Compute scaled dot-product attention for 3 tokens with d_k = 2: weights = softmax(Q·Kᵀ / √d_k) row-wise, "
        "output = weights·V." + (" Apply a causal mask (token i attends only to tokens j <= i)." if causal else "")
        + f"\nQ = {js(Q)}\nK = {js(K)}\nV = {js(V)}",
        '{"attention_weights": 3×3 list of rows, "output": 3×2 list of rows}',
        {"attention_weights": w, "output": w @ V})


@module(27, "Autoencoder", "Deep learning")
def m27(rng):
    rows = sample(rng, 3)
    x = np.c_[zs(rows, "torque_nm"), zs(rows, "tool_wear_min")]
    we, be = w1(rng, 2), w1(rng, None)
    wd, bd = w1(rng, 2), w1(rng, 2, -0.5, 0.5)
    code = x @ we + be
    mse = ((x - (code[:, None] * wd + bd)) ** 2).mean(axis=1)
    thr = round(float(np.median(mse)) + 0.05, 2)
    if np.any(np.abs(mse - thr) < 0.02):
        return None
    return Problem(
        "A linear autoencoder compresses [standardised torque, standardised tool wear] to a 1-D code: "
        f"code = w_e·x + b_e, reconstruction x̂ = code·w_d + b_d. w_e = {js(we)}, b_e = {be}, w_d = {js(wd)}, b_d = {js(bd)}. "
        f"Reconstruction error = mean over the 2 features of (x − x̂)². Flag an anomaly when the error > {thr}.\n"
        f"Inputs:\n{indexed(x)}",
        '{"codes": [number per input], "errors": [number per input], "anomalies": [true/false per input]}',
        {"codes": code, "errors": mse, "anomalies": mse > thr})

### Evolutionary (module 30)

In [ ]:
@module(30, "Genetic Algorithm", "Evolutionary")
def m30(rng):
    pop = rng.integers(0, 2, (4, 8))
    value = rng.integers(-3, 6, 8)
    fit = pop @ value
    if fit[0] == fit[2] or fit[1] == fit[3]:
        return None
    p1, p2 = (0 if fit[0] > fit[2] else 2), (1 if fit[1] > fit[3] else 3)
    k = int(rng.integers(2, 7))
    c1 = np.r_[pop[p1][:k], pop[p2][k:]]
    c2 = np.r_[pop[p2][:k], pop[p1][k:]]
    m1, m2 = (int(v) for v in rng.integers(0, 8, 2))
    c1[m1] ^= 1
    c2[m2] ^= 1
    bits = lambda b: "".join(map(str, b))
    return Problem(
        "One generation of a genetic algorithm evolving 8-week maintenance schedules (bit i = 1 means maintain in week i). "
        f"Fitness = Σ value_i·bit_i with value = {js(value)}.\nPopulation:\n"
        + "\n".join(f"  {i}: {bits(p)}" for i, p in enumerate(pop))
        + "\nSelection: tournament 0 vs 2 gives parent A, tournament 1 vs 3 gives parent B (higher fitness wins). "
        f"Single-point crossover after position {k}: child1 = A[:{k}] + B[{k}:], child2 = B[:{k}] + A[{k}:] "
        f"(0-based slicing). Mutation: flip bit {m1} of child1 and bit {m2} of child2.",
        '{"fitness": [number per individual], "parents": [index of A, index of B], '
        '"children": ["8-bit string", "8-bit string"], "children_fitness": [number, number]}',
        {"fitness": fit, "parents": [p1, p2], "children": [bits(c1), bits(c2)], "children_fitness": [c1 @ value, c2 @ value]})

### Item generation

In [ ]:
SYSTEM_PROMPT = (
    "You are an expert in machine learning and numerical computation. Carry out the requested computation exactly as "
    "specified. You may reason step by step, but the last thing in your reply must be the final answer as a single "
    "JSON object, with numbers given to at least 4 significant figures."
)


@dataclass
class TaskItem:
    item_id: str
    module: int
    prompt: str
    expected: dict
    meta: dict


def make_items(mod: Module, n: int) -> list[TaskItem]:
    rng = np.random.default_rng([SEED, mod.num])
    items, attempts = [], 0
    while len(items) < n:
        attempts += 1
        if attempts > 500 * n:
            raise RuntimeError(f"Module {mod.key}: could not generate unambiguous items")
        prob = mod.generate(rng)
        if prob is None:
            continue
        prompt = (f"Module {mod.num} — {mod.name}\n\n{prob.body}\n\n"
                  f"Final answer format (JSON): {prob.answer}")
        items.append(TaskItem(f"m{mod.num:02d}-{len(items):02d}", mod.num, prompt, to_py(prob.expected), prob.meta))
    return items


ITEMS: list[TaskItem] = [it for mod in sorted(MODULES.values(), key=lambda m: m.num) for it in make_items(mod, ITEMS_PER_MODULE)]
ITEM_BY_ID = {it.item_id: it for it in ITEMS}

assert sorted(MODULES) == list(range(1, 31)), "all 30 modules must be registered"
print(f"Generated {len(ITEMS)} items: {len(MODULES)} modules × {ITEMS_PER_MODULE}")
print("\nExample prompt:\n\n" + ITEMS[ITEMS_PER_MODULE * 10].prompt)

## 4 · Model registry & adapters

* **Anthropic** models are called through the official `anthropic` SDK.
* **Every other provider** is reached through its OpenAI-compatible Chat Completions endpoint via the `openai` SDK,
  so one adapter covers OpenAI, Google Gemini, OpenRouter (hundreds of models behind one key), Groq, Together,
  Mistral, DeepSeek, xAI and local Ollama / vLLM / LM Studio.

A model is benchmarked only when its provider's API key is set. Non-Anthropic model IDs change often, so they are
supplied through environment variables rather than hard-coded:

```bash
export ANTHROPIC_API_KEY=...                       # Claude models below
export OPENAI_API_KEY=...;     export OPENAI_MODELS="model-a,model-b"
export GEMINI_API_KEY=...;     export GEMINI_MODELS="model-c"
export OPENROUTER_API_KEY=...; export OPENROUTER_MODELS="vendor/model-d,vendor/model-e"
export OLLAMA_MODELS="llama-local"                  # local, no key
```

Optional pricing (USD per 1M input/output tokens): `MODEL_PRICES='{"vendor/model-d": [0.5, 1.5]}'`.

Fairness notes:
* No sampling parameters are sent; each model runs at its provider's defaults.
* Server-side refusal **fallbacks are not enabled** for Claude models, because a fallback would let a different
  model answer. Refusals count as failed items.
* The **calibration baselines** are not LLMs: `oracle` returns the reference answer (should score 1.0),
  `noisy-oracle` corrupts ~30% of answers, and `null` returns `{}` (should score 0).

In [ ]:
@dataclass
class ModelSpec:
    key: str                       # unique label used in the leaderboard
    provider: str                  # "anthropic" | "openai_compatible" | "baseline"
    model_id: str
    input_usd_per_mtok: float = float("nan")
    output_usd_per_mtok: float = float("nan")
    base_url: str | None = None
    api_key_env: str | None = None
    token_param: str = "max_tokens"
    extra: dict = field(default_factory=dict)


@dataclass
class Completion:
    text: str
    input_tokens: int = 0
    output_tokens: int = 0
    latency_s: float = 0.0
    error: str | None = None


# Claude models: (model id, input $/MTok, output $/MTok). Override with ANTHROPIC_MODELS="id1,id2".
ANTHROPIC_DEFAULTS = [
    ("claude-fable-5-1", 10.00, 50.00),
    ("claude-opus-5-5", 4.00, 20.00),
    ("claude-sonnet-5-5", 2.00, 10.00),
    ("claude-haiku-5-5", 0.10, 0.50),
]

OPENAI_COMPATIBLE_PROVIDERS = {
    # env prefix: (base_url, api-key env var or None for keyless local servers, output-token parameter name)
    "OPENAI": ("https://api.openai.com/v1", "OPENAI_API_KEY", "max_completion_tokens"),
    "GEMINI": ("https://generativelanguage.googleapis.com/v1beta/openai/", "GEMINI_API_KEY", "max_tokens"),
    "OPENROUTER": ("https://openrouter.ai/api/v1", "OPENROUTER_API_KEY", "max_tokens"),
    "GROQ": ("https://api.groq.com/openai/v1", "GROQ_API_KEY", "max_tokens"),
    "TOGETHER": ("https://api.together.xyz/v1", "TOGETHER_API_KEY", "max_tokens"),
    "MISTRAL": ("https://api.mistral.ai/v1", "MISTRAL_API_KEY", "max_tokens"),
    "DEEPSEEK": ("https://api.deepseek.com/v1", "DEEPSEEK_API_KEY", "max_tokens"),
    "XAI": ("https://api.x.ai/v1", "XAI_API_KEY", "max_tokens"),
    "OLLAMA": (os.environ.get("OLLAMA_BASE_URL", "http://localhost:11434/v1"), None, "max_tokens"),
}


def env_list(name: str) -> list[str]:
    return [m.strip() for m in os.environ.get(name, "").split(",") if m.strip()]


def anthropic_credentials_available() -> bool:
    return bool(os.environ.get("ANTHROPIC_API_KEY") or os.environ.get("ANTHROPIC_AUTH_TOKEN")
                or os.environ.get("ANTHROPIC_PROFILE") or (Path.home() / ".config" / "anthropic").is_dir())


def build_registry() -> list[ModelSpec]:
    specs: list[ModelSpec] = []
    prices = json.loads(os.environ.get("MODEL_PRICES", "{}"))
    if anthropic_credentials_available():
        defaults = {m: (i, o) for m, i, o in ANTHROPIC_DEFAULTS}
        for model_id in env_list("ANTHROPIC_MODELS") or list(defaults):
            i, o = prices.get(model_id, defaults.get(model_id, (float("nan"), float("nan"))))
            specs.append(ModelSpec(model_id, "anthropic", model_id, i, o))
    for prefix, (base_url, key_env, token_param) in OPENAI_COMPATIBLE_PROVIDERS.items():
        if key_env and not os.environ.get(key_env):
            continue
        for model_id in env_list(f"{prefix}_MODELS"):
            i, o = prices.get(model_id, (float("nan"), float("nan")))
            specs.append(ModelSpec(f"{prefix.lower()}:{model_id}", "openai_compatible", model_id, i, o,
                                   base_url=base_url, api_key_env=key_env, token_param=token_param))
    if INCLUDE_BASELINES:
        specs += [ModelSpec(f"baseline:{b}", "baseline", b, 0.0, 0.0) for b in ["oracle", "noisy-oracle", "null"]]
    return specs


# ------------------------------------------------------------------ adapters
class AnthropicAdapter:
    def __init__(self, spec: ModelSpec):
        import anthropic
        self.anthropic = anthropic
        self.spec = spec
        self.client = anthropic.Anthropic(timeout=REQUEST_TIMEOUT_S, max_retries=3)

    def complete(self, item: TaskItem) -> Completion:
        a = self.anthropic
        t0 = time.perf_counter()
        try:
            resp = self.client.messages.create(
                model=self.spec.model_id,
                max_tokens=MAX_OUTPUT_TOKENS,
                system=SYSTEM_PROMPT,
                messages=[{"role": "user", "content": item.prompt}],
                **self.spec.extra,  # e.g. {"output_config": {"effort": "low"}}
            )
        except a.RateLimitError as e:
            return Completion("", latency_s=time.perf_counter() - t0, error=f"rate_limited: {e.message}")
        except a.APIStatusError as e:
            return Completion("", latency_s=time.perf_counter() - t0, error=f"http_{e.status_code}: {e.message}")
        except a.APIConnectionError as e:
            return Completion("", latency_s=time.perf_counter() - t0, error=f"connection: {e}")
        latency = time.perf_counter() - t0
        text = "".join(b.text for b in resp.content if b.type == "text")
        error = "refusal" if resp.stop_reason == "refusal" else ("max_tokens" if resp.stop_reason == "max_tokens" else None)
        return Completion(text, resp.usage.input_tokens, resp.usage.output_tokens, latency, error)


class OpenAICompatibleAdapter:
    def __init__(self, spec: ModelSpec):
        import openai
        self.openai = openai
        self.spec = spec
        api_key = os.environ.get(spec.api_key_env) if spec.api_key_env else "not-needed"
        self.client = openai.OpenAI(base_url=spec.base_url, api_key=api_key, timeout=REQUEST_TIMEOUT_S, max_retries=3)

    def complete(self, item: TaskItem) -> Completion:
        t0 = time.perf_counter()
        try:
            resp = self.client.chat.completions.create(
                model=self.spec.model_id,
                messages=[{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": item.prompt}],
                **{self.spec.token_param: MAX_OUTPUT_TOKENS},
                **self.spec.extra,
            )
        except self.openai.APIStatusError as e:
            return Completion("", latency_s=time.perf_counter() - t0, error=f"http_{e.status_code}: {e.message}")
        except self.openai.APIConnectionError as e:
            return Completion("", latency_s=time.perf_counter() - t0, error=f"connection: {e}")
        latency = time.perf_counter() - t0
        choice = resp.choices[0] if resp.choices else None
        text = (choice.message.content or "") if choice else ""
        usage = resp.usage
        error = "max_tokens" if choice and choice.finish_reason == "length" else None
        return Completion(text, getattr(usage, "prompt_tokens", 0) or 0, getattr(usage, "completion_tokens", 0) or 0, latency, error)


class BaselineAdapter:
    """Non-LLM calibration baselines used to sanity-check scoring and ranking."""

    def __init__(self, spec: ModelSpec):
        self.spec = spec

    def complete(self, item: TaskItem) -> Completion:
        if self.spec.model_id == "null":
            return Completion("{}")
        answer = json.loads(json.dumps(item.expected))
        if self.spec.model_id == "noisy-oracle":
            rng = random.Random(f"{SEED}-{item.item_id}")
            if rng.random() < 0.3:
                key = rng.choice(list(answer))
                answer[key] = perturb(answer[key])
        return Completion("Working omitted.\n" + json.dumps(answer))


def perturb(v):
    """Corrupt the first leaf of a value so it is clearly wrong."""
    if isinstance(v, bool):
        return not v
    if isinstance(v, (int, float)):
        return v * 1.5 + 1
    if isinstance(v, str):
        return v[::-1] + "x"
    if isinstance(v, list) and v:
        return [perturb(v[0])] + v[1:]
    return v


ADAPTERS = {"anthropic": AnthropicAdapter, "openai_compatible": OpenAICompatibleAdapter, "baseline": BaselineAdapter}

REGISTRY = build_registry()
pd.DataFrame([{"model": s.key, "provider": s.provider, "in $/MTok": s.input_usd_per_mtok,
               "out $/MTok": s.output_usd_per_mtok} for s in REGISTRY])

## 5 · Execution engine

Runs every item against every registered model with a thread pool. Each response is cached on disk under a hash of
(model, settings, system prompt, prompt), so interrupted or repeated runs only pay for missing items. Transport
failures (HTTP errors, rate limits, connection drops) are not cached and are retried on the next run.

In [ ]:
def cache_path(spec: ModelSpec) -> Path:
    return CACHE_DIR / (re.sub(r"[^A-Za-z0-9_.-]+", "_", spec.key) + ".jsonl")


def cache_key(spec: ModelSpec, item: TaskItem) -> str:
    payload = json.dumps([spec.provider, spec.model_id, spec.extra, MAX_OUTPUT_TOKENS, SYSTEM_PROMPT, item.prompt], sort_keys=True)
    return hashlib.sha256(payload.encode()).hexdigest()


def load_cache(spec: ModelSpec) -> dict[str, dict]:
    path = cache_path(spec)
    if not path.exists():
        return {}
    with path.open() as f:
        return {rec["key"]: rec for rec in map(json.loads, f)}


def run_model(spec: ModelSpec, items: list[TaskItem]) -> list[dict]:
    adapter = ADAPTERS[spec.provider](spec)
    cache = {} if spec.provider == "baseline" else load_cache(spec)
    records, todo = [], []
    for item in items:
        k = cache_key(spec, item)
        (records.append(cache[k]) if k in cache else todo.append((k, item)))

    def call(k: str, item: TaskItem) -> dict:
        c = adapter.complete(item)
        return {"key": k, "model": spec.key, "item_id": item.item_id, "text": c.text, "input_tokens": c.input_tokens,
                "output_tokens": c.output_tokens, "latency_s": c.latency_s, "error": c.error}

    if todo:
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool, cache_path(spec).open("a") as f:
            futures = [pool.submit(call, k, it) for k, it in todo]
            for fut in as_completed(futures):
                rec = fut.result()
                records.append(rec)
                if spec.provider != "baseline" and not (rec["error"] or "").startswith(("http_", "connection", "rate_limited")):
                    f.write(json.dumps(rec) + "\n")
    print(f"{spec.key:40s} {len(items) - len(todo):4d} cached, {len(todo):4d} new")
    return records


RAW: list[dict] = []
for spec in REGISTRY:
    RAW += run_model(spec, ITEMS)

## 6 · Scoring

The **final answer** is the last complete JSON object in the reply, so models may show their working first.
Each top-level answer field is scored in [0, 1] and the item score is the mean over fields:

* numbers match within max(2% relative, 0.01 absolute); booleans, strings and integer labels must match exactly;
* lists and matrices score the fraction of elements that match (missing elements count as wrong);
* `partition` fields (clusterings) match when they group the same indices, whatever the cluster numbering;
* `set` fields ignore order; `sign_invariant` fields (PCA directions) accept v or −v.

In [ ]:
def extract_final_json(text: str) -> tuple[Any, bool]:
    """Return (last complete top-level JSON object in the text, found?)."""
    text = (text or "").strip()
    try:
        value = json.loads(text)
        return value, isinstance(value, dict)
    except (json.JSONDecodeError, TypeError):
        pass
    decoder, best = json.JSONDecoder(), None
    for m in re.finditer(r"\{", text):
        try:
            value, end = decoder.raw_decode(text, m.start())
        except json.JSONDecodeError:
            continue
        if isinstance(value, dict) and (best is None or end > best[1]):
            best = (value, end)
    return (best[0], True) if best else (None, False)


def num_close(p, e) -> bool:
    if isinstance(p, bool) or p is None:
        return False
    try:
        return math.isclose(float(p), float(e), rel_tol=REL_TOL, abs_tol=ABS_TOL)
    except (TypeError, ValueError):
        return False


def leaf_match(p, e) -> tuple[int, int]:
    """(correct leaves, total leaves) of prediction p against expected e."""
    if isinstance(e, bool):
        ok = p is e or (type(p) is int and p in (0, 1) and bool(p) == e) or (isinstance(p, str) and p.lower() == str(e).lower())
        return int(ok), 1
    if isinstance(e, int):
        return int(not isinstance(p, bool) and isinstance(p, (int, float)) and float(p) == e), 1
    if isinstance(e, float):
        return int(num_close(p, e)), 1
    if isinstance(e, str):
        return int(isinstance(p, (str, int, float)) and str(p).strip().lower() == e.lower()), 1
    if isinstance(e, list):
        c = t = 0
        for i, ei in enumerate(e):
            ci, ti = leaf_match(p[i] if isinstance(p, list) and i < len(p) else None, ei)
            c, t = c + ci, t + ti
        return c, max(t, 1)
    if isinstance(e, dict):
        c = t = 0
        for k, ei in e.items():
            ci, ti = leaf_match(p.get(k) if isinstance(p, dict) else None, ei)
            c, t = c + ci, t + ti
        return c, max(t, 1)
    return int(p == e), 1


def as_index_set(v) -> frozenset | None:
    try:
        return frozenset(int(i) for i in v)
    except (TypeError, ValueError):
        return None


def score_field(key: str, p, e, meta: dict) -> float:
    if key in meta.get("partition", []):
        groups = [as_index_set(g) for g in p] if isinstance(p, list) else None
        return float(groups is not None and None not in groups and set(groups) == {frozenset(g) for g in e})
    if key in meta.get("set", []):
        return float(isinstance(p, list) and as_index_set(p) == frozenset(e))
    if key in meta.get("sign_invariant", []):
        neg = [-v for v in e]
        return max(leaf_match(p, e)[0], leaf_match(p, neg)[0]) / len(e)
    c, t = leaf_match(p, e)
    return c / t


def score_item(pred, item: TaskItem) -> float:
    if not isinstance(pred, dict):
        return 0.0
    return float(np.mean([score_field(k, pred.get(k), e, item.meta) for k, e in item.expected.items()]))


SPEC_BY_KEY = {s.key: s for s in REGISTRY}


def score_record(rec: dict) -> dict:
    item = ITEM_BY_ID[rec["item_id"]]
    mod = MODULES[item.module]
    pred, found = extract_final_json(rec["text"])
    spec = SPEC_BY_KEY[rec["model"]]
    cost = (rec["input_tokens"] * spec.input_usd_per_mtok + rec["output_tokens"] * spec.output_usd_per_mtok) / 1e6
    return {"model": rec["model"], "provider": spec.provider, "module": mod.key, "family": mod.family,
            "item_id": item.item_id, "score": score_item(pred, item) if found else 0.0, "answer_found": found,
            "fully_correct": found and score_item(pred, item) == 1.0, "error": rec["error"], "latency_s": rec["latency_s"],
            "input_tokens": rec["input_tokens"], "output_tokens": rec["output_tokens"], "cost_usd": cost}


results = pd.DataFrame([score_record(r) for r in RAW])
results.pivot_table(index="module", columns="model", values="score").round(2)

## 7 · Ranking

* **Composite score**: weighted mean of the 30 per-module mean scores (`MODULE_WEIGHTS`, default equal).
* **Family scores**: mean module score within Supervised / Unsupervised / Reinforcement / Deep learning / Evolutionary.
* **95% CI and rank probabilities**: stratified bootstrap. Each replicate resamples items *within each module*,
  using the same items for every model so comparisons are paired, then recomputes composites and ranks.
* **Bradley–Terry rating**: every item is a match between each pair of models (higher item score wins, equal
  scores tie). Strengths are fit with the MM algorithm and shown on an Elo-like scale (1000 = average).
* **Tier**: a new tier starts when a model's CI upper bound falls below the CI lower bound of the tier's leader.
  Differences within a tier are not statistically meaningful at this sample size.

In [ ]:
module_keys = [MODULES[n].key for n in sorted(MODULES)]
weights = pd.Series({MODULES[n].key: MODULE_WEIGHTS.get(n, 1.0) for n in sorted(MODULES)})
weights /= weights.sum()
models = sorted(results["model"].unique())
families = list(dict.fromkeys(MODULES[n].family for n in sorted(MODULES)))

# score matrix per module: rows = items, cols = models (paired design)
score_mats = {m: results[results.module == m].pivot(index="item_id", columns="model", values="score")[models].fillna(0.0)
              for m in module_keys}
module_means = pd.DataFrame({m: score_mats[m].mean() for m in module_keys})            # models × modules
point = (module_means * weights).sum(axis=1)

rng = np.random.default_rng(SEED)
arrays = [(weights[m], score_mats[m].to_numpy()) for m in module_keys]
boot = np.empty((BOOTSTRAP_SAMPLES, len(models)))
for b in range(BOOTSTRAP_SAMPLES):
    boot[b] = sum(w * a[rng.integers(0, len(a), len(a))].mean(axis=0) for w, a in arrays)
ci_lo, ci_hi = np.percentile(boot, 2.5, axis=0), np.percentile(boot, 97.5, axis=0)
boot_ranks = (-boot).argsort(axis=1).argsort(axis=1) + 1
p_first = (boot_ranks == 1).mean(axis=0)


def bradley_terry(mats: dict[str, pd.DataFrame], iters: int = 500) -> pd.Series:
    n = len(models)
    wins = np.zeros((n, n))
    for m in mats.values():
        a = m.to_numpy()
        for i in range(n):
            for j in range(i + 1, n):
                ties = 0.5 * (a[:, i] == a[:, j]).sum()
                wins[i, j] += (a[:, i] > a[:, j]).sum() + ties
                wins[j, i] += (a[:, j] > a[:, i]).sum() + ties
    wins += 0.5 * (1 - np.eye(n))  # weak prior keeps undefeated / winless models finite
    games = wins + wins.T
    p = np.ones(n)
    for _ in range(iters):
        p = wins.sum(axis=1) / (games / (p[:, None] + p[None, :])).sum(axis=1)
        p /= np.exp(np.log(p).mean())
    return pd.Series(1000 + 400 * np.log10(p), index=models)


family_scores = pd.DataFrame({f: module_means[[MODULES[n].key for n in MODULES if MODULES[n].family == f]].mean(axis=1)
                              for f in families})
agg = results.groupby("model").agg(
    items=("score", "size"), fully_correct_rate=("fully_correct", "mean"), answer_found_rate=("answer_found", "mean"),
    error_rate=("error", lambda s: s.notna().mean()), latency_p50_s=("latency_s", "median"),
    cost_usd=("cost_usd", "sum"), mean_output_tokens=("output_tokens", "mean"),
)

leaderboard = (pd.DataFrame({"composite": point, "ci_low": ci_lo, "ci_high": ci_hi, "p_rank1": p_first,
                             "bt_rating": bradley_terry(score_mats)}, index=models)
               .join(family_scores).join(agg))
leaderboard["cost_per_100_items"] = leaderboard["cost_usd"] / leaderboard["items"] * 100
leaderboard = leaderboard.sort_values(["composite", "bt_rating"], ascending=False)
leaderboard.insert(0, "rank", range(1, len(leaderboard) + 1))

tier, tier_floor, tiers = 0, None, []
for _, row in leaderboard.iterrows():
    if tier_floor is None or row.ci_high < tier_floor:
        tier, tier_floor = tier + 1, row.ci_low
    tiers.append(tier)
leaderboard.insert(1, "tier", tiers)

show = ["rank", "tier", "composite", "ci_low", "ci_high", "p_rank1", "bt_rating", *families,
        "fully_correct_rate", "answer_found_rate", "error_rate", "latency_p50_s", "cost_per_100_items"]
leaderboard[show].round(3)

## 8 · Reporting

In [ ]:
INK, MUTED, GRID, BAR, SURFACE = "#0b0b0b", "#898781", "#e1e0d9", "#2a78d6", "#fcfcfb"
plt.rcParams.update({"font.size": 10, "axes.edgecolor": "#c3c2b7", "axes.labelcolor": INK, "xtick.color": MUTED,
                     "ytick.color": INK, "axes.spines.top": False, "axes.spines.right": False,
                     "figure.facecolor": SURFACE, "axes.facecolor": SURFACE})

# 8.1 Leaderboard with bootstrap 95% CI
lb = leaderboard.iloc[::-1]
fig, ax = plt.subplots(figsize=(8, 0.45 * len(lb) + 1.2))
ax.barh(lb.index, lb.composite, color=BAR, height=0.6)
ax.errorbar(lb.composite, lb.index, xerr=[lb.composite - lb.ci_low, lb.ci_high - lb.composite],
            fmt="none", ecolor=INK, elinewidth=1, capsize=3)
for y, (_, r) in enumerate(lb.iterrows()):
    ax.text(min(r.ci_high + 0.02, 1.02), y, f"{r.composite:.3f}  (tier {r.tier})", va="center", color=INK, fontsize=9)
ax.set_xlim(0, 1.2)
ax.set_xticks(np.linspace(0, 1, 6))
ax.set_xlabel("Composite score across 30 algorithm modules, 95% bootstrap CI")
ax.set_title("LLM ranking — structured data processing", loc="left", color=INK, fontsize=12)
ax.grid(axis="x", color=GRID, linewidth=0.8)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "leaderboard.png", dpi=150)
plt.show()

# 8.2 Module × model heatmap (single-hue sequential scale), grouped by family
order = [MODULES[n].key for f in families for n in sorted(MODULES) if MODULES[n].family == f]
heat = module_means.loc[leaderboard.index, order].T
fig, ax = plt.subplots(figsize=(1.3 * len(heat.columns) + 4, 0.32 * len(heat) + 1.5))
im = ax.imshow(heat.to_numpy(), cmap="Blues", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(len(heat.columns)), heat.columns, rotation=30, ha="right")
ax.set_yticks(range(len(heat)), [f"{k}  ·  {MODULES[int(k[:2])].family}" for k in heat.index], fontsize=8.5)
for (i, j), v in np.ndenumerate(heat.to_numpy()):
    ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=8, color="#ffffff" if v > 0.6 else INK)
boundaries = np.cumsum([sum(MODULES[n].family == f for n in MODULES) for f in families])[:-1]
for bnd in boundaries:
    ax.axhline(bnd - 0.5, color=SURFACE, linewidth=3)
ax.tick_params(length=0)
for s in ax.spines.values():
    s.set_visible(False)
fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02, label="Mean item score")
ax.set_title("Score by algorithm module", loc="left", color=INK, fontsize=12)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "module_heatmap.png", dpi=150)
plt.show()

# 8.3 Cost vs quality (only models with known, non-zero cost)
priced = leaderboard[leaderboard.cost_per_100_items.notna() & (leaderboard.cost_per_100_items > 0)]
if len(priced):
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.scatter(priced.cost_per_100_items, priced.composite, s=60, color=BAR, edgecolor=SURFACE, linewidth=2, zorder=3)
    pareto, best = [], -1.0
    for name, r in priced.sort_values("cost_per_100_items").iterrows():
        if r.composite > best:
            pareto.append(name); best = r.composite
    for name, r in priced.iterrows():
        ax.annotate(name + (" ★" if name in pareto else ""), (r.cost_per_100_items, r.composite),
                    xytext=(6, 4), textcoords="offset points", fontsize=9, color=INK)
    ax.set_xscale("log")
    ax.set_xlabel("Cost per 100 items (USD, log scale)")
    ax.set_ylabel("Composite score")
    ax.set_title("Cost vs quality (★ = Pareto-optimal)", loc="left", color=INK, fontsize=12)
    ax.grid(color=GRID, linewidth=0.8)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "cost_vs_quality.png", dpi=150)
    plt.show()
else:
    print("No priced models in this run — cost/quality chart skipped.")

In [ ]:
# 8.4 Best model per module and weakest modules per real model (baselines excluded)
real_models = [m for m in leaderboard.index if SPEC_BY_KEY[m].provider != "baseline"]
if real_models:
    mm = module_means.loc[real_models, module_keys]
    display(pd.DataFrame({"best_model": mm.idxmax(), "best_score": mm.max().round(3), "mean_score": mm.mean().round(3)}))
    display(pd.DataFrame({m: mm.loc[m].nsmallest(3).index.tolist() for m in real_models}, index=["weakest 1", "weakest 2", "weakest 3"]))
else:
    print("Only calibration baselines ran. Set API keys (Section 4) to rank real models.")

# 8.5 Export
results.to_csv(RESULTS_DIR / "item_results.csv", index=False)
leaderboard.join(module_means).to_csv(RESULTS_DIR / "leaderboard.csv")
with open(RESULTS_DIR / "leaderboard.md", "w") as f:
    f.write("# LLM ranking — structured data processing (30 ML algorithm modules)\n\n")
    f.write(f"Data: {DATA_SOURCE}. Items: {len(ITEMS)} ({ITEMS_PER_MODULE} per module). "
            f"Bootstrap samples: {BOOTSTRAP_SAMPLES}.\n\n")
    try:
        f.write(leaderboard[show].round(3).to_markdown())  # needs the optional `tabulate` package
    except ImportError:
        f.write("```\n" + leaderboard[show].round(3).to_string() + "\n```")
    f.write("\n")
print(f"Wrote results to {RESULTS_DIR.resolve()}")

## Interpreting and extending

* **Read tiers before ranks.** With 5 items per module (150 per model), small composite differences are often
  noise; raise `ITEMS_PER_MODULE` until the CIs separate the models you care about.
* **Family scores show where a model is reliable.** A model may be excellent at supervised-learning arithmetic but
  weak at multi-step procedures such as DBSCAN, value iteration or LSTM recurrences.
* **`fully_correct_rate`** is the share of items with every field right: the strictest view, and the one that
  matters when an LLM's output feeds a pipeline unchecked.
* **Re-weighting**: set `MODULE_WEIGHTS` (e.g. `{2: 3.0, 11: 3.0}`) to emphasise the algorithms your workload uses.
* **Adding a module**: write a generator returning a `Problem` and decorate it with `@module(num, name, family)`;
  generation, scoring, ranking and reporting pick it up automatically.
* **Per-model settings**: `ModelSpec.extra` is passed through to the API call, e.g.
  `{"output_config": {"effort": "high"}}` for a Claude model, to rank one model at several effort levels.